## **Курсовая работа 3-го курса ОП "Компьютерные науки и технологии" в НИУ ВШЭ-НН**
### **Название: Мониторинг репутации компаний через анализ упоминаний в сети**

**Срок выполнения:** с ..2026 г. по 21.05.2026 г.  
**Выполнил данную часть:** Иван Гайфиев (23КНТ-6)


### **Описание задания:**
Требуется разработать набор алгоритмов, задачей которых является NLP-анализ входных данных (упоминаний о компаниях) с целью формулировки основных тем, выявления наиболее часто упоминаемых слов и определения лучших и худших сторон компании на основе отзывов.

#### **Основные этапы:**
1) Подключение к базе данных Supabase и загрузка из неё набор упоминаний для последующего анализа в формате csv
2) Предобработать датафрейм, оставив данные, соответствующие определённым условиям
3) Разработать алгоритм по моделлированию ключевых тем
4) Разработать алгоритм по формированию облака слов
5) Разработать алгоритм по составлению списков позитивных и негативных аспектов
6) Выполнить анализ исходных данных с помощью этих алгоритмов и выгрузить результаты в БД

#### **Полезные материалы:**
1) Доска MIRO с описанием системы и её компонентов - https://miro.com/app/board/uXjVHW0IYa8=/


### **Выполнение задания:**


#### **Этап 1: Подключение к БД и загрузка данных**
**Задание:**
Подключиться к базе данных Supabase и загрузить из неё набор упоминаний для последующего анализа в формате csv

**Выполнение:**  


In [11]:
! pip install supabase google.colab scikit-learn nltk bertopic sentence-transformers natasha -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 88.6 MB/s eta 0:00:00


In [12]:
import os
import pandas as pd
import numpy as np
from google.colab import drive
from supabase import create_client, Client
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk
from bertopic import BERTopic
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    Doc
)

In [20]:
# Loading data
def mount_file():
    drive.mount('/content/drive')
    config_file = "/content/drive/MyDrive/Labs in Colab /Course work 3rd year/config.txt"
    return config_file

def get_config():
    config_file = mount_file()
    with open (config_file, 'r') as f:
        SUPABASE_URL = f.readline().strip()
        SUPABASE_KEY = f.readline().strip()
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    return supabase

def import_data(table_name, company_id, output_file="output.csv"):
    supabase = get_config()

    try:
        response = supabase.table(table_name).select("mention_id, text").eq("company_id", company_id).execute()

        if not response.data:
            print(f"Нет данных для company_id={company_id}.")
            return None

        df = pd.DataFrame(response.data)

        df.to_csv(output_file, index=False, encoding="utf-8-sig")

        print(f"Сохранено {len(df)} строк в {output_file}")

        return df

    except Exception as e:
        print("Ошибка при получении данных:")
        print(e)

df = import_data("mentions", 5)
print(df.head())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Сохранено 1000 строк в output.csv
   mention_id                                               text
0        4959                             прекрасное приложение.
1        4616                          доставляет раньше посылки
2        4564  бесполезное приложение, не работает, просит вп...
3        4201                                         все холёщё
4        4202                                               good


In [ ]:
# TOPIC MODELLING
# Base algorithm - не использовать в проде

def topics_base(df):
    nltk.download("stopwords")
    russian_stopwords = stopwords.words("russian")
    texts = df["text"].dropna().astype(str).tolist()

    vectorizer = TfidfVectorizer(
        max_features=1000,
        stop_words=russian_stopwords
    )

    X = vectorizer.fit_transform(texts)
    words = vectorizer.get_feature_names_out()

    scores = X.sum(axis=0).A1

    keywords = pd.DataFrame({
        "word": words,
        "score": scores
    }).sort_values("score", ascending=False)

    top_words = keywords.head(20)["word"].tolist()

    topics_df = pd.DataFrame([{
        "method": "TF-IDF",
        "topic": "Общая тема отзывов",
        "keywords": ", ".join(top_words),
        "sentence": f"Основные обсуждаемые темы: {', '.join(top_words[:10])}."
    }])

    return topics_df




In [ ]:
# Middle algorithm

In [21]:
# Advanced algorithm

def topics_advanced(df):
    # оставим отзывы где больше 30 символов и хотя бы 3 пробела
    df_clean = df[
        (df["text"].str.len() > 30) &
        (df["text"].str.count(' ') > 3)
    ].copy()

    docs = df["text"].dropna().astype(str).tolist()

    model = BERTopic(language="multilingual", n_gram_range=(1, 2))
    topics, probs = model.fit_transform(docs)

    doc_embeddings = model.embedding_model.embed(docs) # эмбеддинги
    topic_embeddings = model.topic_embeddings_ # центроиды

    topic_rows =[]
    info = model.get_topic_info()

    for i, topic_id in enumerate(info["Topic"]):
        if topic_id == -1: continue # шум

        # индекс самого центрального/типичного отзыва
        topic_vec = topic_embeddings[i].reshape(1, -1)
        sims = cosine_similarity(topic_vec, doc_embeddings)[0]
        most_representative_idx = np.argmax(sims)

        # самый показательный отзыв (фрагмент)
        representative_sentence = docs[most_representative_idx]

        if len(representative_sentence.split()) < 4: # хотя бы 3-4 слова в теме
            continue

        topic_rows.append({
            "method": "BERTopic_Semantic",
            "trend_text": representative_sentence,
            "relevance": float(sims[most_representative_idx]),
            "source": f"Topic_{topic_id}"
        })

    return pd.DataFrame(topic_rows)

final_topics = topics_advanced(df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
# Saving topics to DB
def upload_topics_db(final_topics, company_id):

    df = final_topics.copy()

    df["company_id"] = company_id
    df["trend_type"] = "general"

    if "relevance" not in df.columns:
        df["relevance"] = 0.8

    supabase = get_config()
    records = df.to_dict(orient="records")

    supabase.table("topic_summary").delete().eq("company_id", company_id).execute()

    for r in records:
        data = {
            "company_id": int(r["company_id"]),
            "trend_text": r.get("trend_text", ""),
            "trend_type": r.get("trend_type", "general"),
            "source": r.get("source", "unknown"),
            "relevance": float(r.get("relevance", 0.8))
        }

        try:
            supabase.table("topic_summary").insert(data).execute()

            print("Inserted:", data["trend_text"][:60])

        except Exception as e:
            print("Error:", e, data)

upload_topics_db(final_topics, 5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inserted: впервые скачала вб, так и подозревала что будет уйма мороки.
Inserted: хорошей магазин отличный товар
Inserted: все супер .мне нравится
Inserted: удобно быстро и качественно


In [26]:
def get_word_frequencies(df, top_n=50):
    segmenter = Segmenter()
    morph_vocab = MorphVocab()
    emb = NewsEmbedding()
    morph_tagger = NewsMorphTagger(emb)

    text = " ".join(df["text"].dropna().astype(str).tolist())

    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    allowed_pos = {'NOUN', 'ADJ', 'VERB'}

    important_words = []

    for token in doc.tokens:
        if token.pos in allowed_pos:
            token.lemmatize(morph_vocab)
            lemma = token.lemma.lower()

            if lemma.isalpha() and len(lemma) > 2:
                important_words.append(lemma)

    word_counts = Counter(important_words)
    most_common = dict(word_counts.most_common(top_n))

    return most_common

cloud_words_dict = get_word_frequencies(df, 30)
cloud_words_dict

{'приложение': 261,
 'товар': 184,
 'хороший': 105,
 'деньга': 105,
 'поддержка': 94,
 'работать': 92,
 'заказ': 76,
 'мочь': 73,
 'удобный': 62,
 'доставка': 60,
 'нет': 57,
 'спасибо': 54,
 'подписка': 53,
 'день': 52,
 'обновление': 52,
 'нравиться': 50,
 'писать': 46,
 'быть': 43,
 'карта': 43,
 'раз': 43,
 'кошелек': 42,
 'отличный': 41,
 'отзыв': 39,
 'отвечать': 39,
 'нужный': 38,
 'пользоваться': 36,
 'хотеть': 36,
 'сам': 35,
 'вопрос': 34,
 'цена': 33}

In [27]:
def upload_wordcloud_to_db(word_freq_dict, company_id):

    supabase = get_config()
    records =[
        {
            "company_id": int(company_id),
            "word": str(word),
            "frequency": int(freq)
        }
        for word, freq in word_freq_dict.items()
    ]

    try:
        supabase.table("wordcloud_cache").delete().eq("company_id", company_id).execute()
        response = supabase.table("wordcloud_cache").insert(records).execute()
        print(f"Успешно загружено {len(records)} слов для компании {company_id}")

    except Exception as e:
        print(f"Ошибка при загрузке облака слов в БД: {e}")
        return None

upload_wordcloud_to_db(cloud_words_dict, 5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Успешно загружено 30 слов для компании 5
